In [1]:
import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from sklearn.metrics import classification_report, confusion_matrix

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    # 2. Get the number of GPUs
    device_count = torch.cuda.device_count()
    print(f"Number of GPUs: {device_count}")

    # 3. Get the name of the current GPU
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    print(f"Current Device Name: {device_name}")

c:\Users\mehla\ML_Learning\myvenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA Available: True
Number of GPUs: 1
Current Device Name: NVIDIA GeForce RTX 3090


In [2]:
from datasets import load_dataset

emotions_data = load_dataset("emotion")

In [3]:
emotions_data

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [4]:
checkpoint = "princeton-nlp/sup-simcse-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

c:\Users\mehla\ML_Learning\myvenv\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mehla\.cache\huggingface\hub\models--princeton-nlp--sup-simcse-bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
def tokenize_function(parameter):
  return tokenizer(parameter["text"], truncation=True)


tokenized_dataset = emotions_data.map(tokenize_function, batched=True)
tokenized_dataset

Map: 100%|██████████| 2000/2000 [00:00<00:00, 51274.48 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [6]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

In [7]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [8]:
tokenized_dataset = tokenized_dataset.rename_column("label","labels")

In [9]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
})

In [10]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [11]:
emotions_data["train"].features

{'text': Value('string'),
 'label': ClassLabel(names=['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'])}

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint , num_labels=6)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39808.58it/s]
BertForSequenceClassification LOAD REPORT from: princeton-nlp/sup-simcse-bert-base-uncased
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
from transformers import TrainingArguments

args = TrainingArguments(
    "test-trainer",
    #evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
)

In [14]:
from transformers import Trainer


# def compute_metrics(eval_preds):
#     metric = evaluate.load("glue", "mrpc")
#     logits, labels = eval_preds
#     predictions = np.argmax(logits, axis=-1)
#     return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    #tokenizer=tokenizer,
)

In [15]:
trainer.train()

Step,Training Loss
500,0.649191
1000,0.300159
1500,0.237172
2000,0.245662
2500,0.159810
3000,0.166717
3500,0.176088
4000,0.163586
4500,0.118397
5000,0.144146


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.63it/s]


TrainOutput(global_step=20000, training_loss=0.09161062710285187, metrics={'train_runtime': 570.7724, 'train_samples_per_second': 280.322, 'train_steps_per_second': 35.04, 'total_flos': 3390685940486016.0, 'train_loss': 0.09161062710285187, 'epoch': 10.0})

In [16]:
predictions = trainer.predict(tokenized_dataset["validation"])

In [17]:
predictions


PredictionOutput(predictions=array([[10.449248  , -2.4466102 , -2.3338296 , -1.7062436 , -2.2370636 ,
        -2.0585063 ],
       [10.405318  , -2.5767522 , -2.4290094 , -1.7331204 , -2.1070879 ,
        -1.8654    ],
       [-3.06874   , 10.05998   ,  0.05997731, -2.8344042 , -2.991213  ,
        -2.0918367 ],
       ...,
       [-2.3817055 , 10.16046   , -1.6904393 , -2.2531743 , -2.5545242 ,
        -2.2095342 ],
       [-2.7058296 , 10.129108  , -0.24968725, -2.8658588 , -2.8267944 ,
        -2.3534303 ],
       [-2.274948  , 10.110466  , -1.7596474 , -2.2635221 , -2.4077945 ,
        -2.36975   ]], shape=(2000, 6), dtype=float32), label_ids=array([0, 0, 2, ..., 1, 1, 1], shape=(2000,)), metrics={'test_loss': 0.4417005479335785, 'test_runtime': 1.9034, 'test_samples_per_second': 1050.743, 'test_steps_per_second': 131.343})

In [18]:
import numpy as np

preds = np.argmax(predictions.predictions, axis=-1)

In [19]:
print(confusion_matrix(preds, emotions_data["validation"]["label"]))
print(classification_report(preds, emotions_data["validation"]["label"]))

[[535   1   2  11   7   1]
 [  1 677  25   4   1   3]
 [  1  19 150   0   0   0]
 [  6   2   0 253   4   0]
 [  7   2   1   7 195   8]
 [  0   3   0   0   5  69]]
              precision    recall  f1-score   support

           0       0.97      0.96      0.97       557
           1       0.96      0.95      0.96       711
           2       0.84      0.88      0.86       170
           3       0.92      0.95      0.94       265
           4       0.92      0.89      0.90       220
           5       0.85      0.90      0.87        77

    accuracy                           0.94      2000
   macro avg       0.91      0.92      0.92      2000
weighted avg       0.94      0.94      0.94      2000



In [20]:
train_preds = trainer.predict(tokenized_dataset["train"])

In [21]:
import numpy as np

train_preds = np.argmax(train_preds.predictions, axis=-1)

In [22]:
print(confusion_matrix(train_preds, emotions_data["train"]["label"]))
print(classification_report(train_preds, emotions_data["train"]["label"]))

[[4663    0    0    1    0    0]
 [   0 5355    8    0    0    1]
 [   0    7 1296    0    0    0]
 [   2    0    0 2155    1    0]
 [   1    0    0    3 1928    0]
 [   0    0    0    0    8  571]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4664
           1       1.00      1.00      1.00      5364
           2       0.99      0.99      0.99      1303
           3       1.00      1.00      1.00      2158
           4       1.00      1.00      1.00      1932
           5       1.00      0.99      0.99       579

    accuracy                           1.00     16000
   macro avg       1.00      1.00      1.00     16000
weighted avg       1.00      1.00      1.00     16000



In [23]:
test_preds = trainer.predict(tokenized_dataset["test"])

In [24]:
test_preds = np.argmax(test_preds.predictions, axis=-1)

print(confusion_matrix(test_preds, emotions_data["test"]["label"]))
print(classification_report(test_preds, emotions_data["test"]["label"]))

[[567   0   2  12   3   2]
 [  1 665  33   2   0   4]
 [  1  24 122   0   0   0]
 [  5   2   2 254   8   0]
 [  7   0   0   7 206  14]
 [  0   4   0   0   7  46]]
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       586
           1       0.96      0.94      0.95       705
           2       0.77      0.83      0.80       147
           3       0.92      0.94      0.93       271
           4       0.92      0.88      0.90       234
           5       0.70      0.81      0.75        57

    accuracy                           0.93      2000
   macro avg       0.87      0.89      0.88      2000
weighted avg       0.93      0.93      0.93      2000

